# IFA2 — Spread-Direct OU Model (Methodology 2)

## Why a second methodology?

The first notebook (`IC_descriptive_stats.ipynb`) follows Abadie & Chamorro (2021): model
`log(P_GB)` and `log(P_FR)` as separate OU+jump processes, then compute revenue as
`|exp(f_GB + X_GB) − exp(f_FR + X_FR)| × capacity × hours`.

This produced P50 revenues of **£305m/yr** against Ofgem-disclosed IFA2 actuals of **£108–188m/yr**.

**Root cause — Jensen's inequality in the log→price-space transformation:**

> E[|exp(X_GB) − exp(X_FR)|]  >>  |exp(E[X_GB]) − exp(E[X_FR])|  when σ is large

Our GB log-price residual σ_d = 0.240 and FR σ_d = 0.421 per day. Both are driven up by the
2021–2022 energy crisis. The inflation factor per process is exp(σ²/2):

- **Abadie (Spain-France, pre-crisis data):** σ ≈ 0.05 → exp(0.05²/2) ≈ **1.001** — negligible
- **This dataset (post-crisis):** σ_FR = 0.42 → exp(0.42²/2) ≈ **1.09**, compounding across both processes

Abadie's methodology is not wrong — it is sound for their lower-volatility context.
Our 2021–2026 sample spans an exceptional volatility regime that breaks the approximation.

## This notebook: model the SPREAD directly (Cartea framework)

Model `S_t = P_GB_t − P_FR_t` (£/MWh, daily, in levels) as OU + jumps:

    S_t = f_S(t) + X_t
    f_S(t)  = OLS deterministic component on spread levels
    X_{t+1} = c + φ·X_t + σ_d·ε_t + J_t

No log transformation → no Jensen's inequality problem. All parameters come from price data.

### Capture ratio
Ofgem revenues are **net of market-related costs** (auction costs, TSO charges, balancing).
We estimate a **capture ratio** = actual mean revenue / theoretical raw spread revenue.
This is derived from the 2022–2025 data and applied as a fixed operational parameter.
The actual revenues then serve as **validation**, not calibration targets.

---
**IFA2 actuals (Ofgem, gross congestion revenues net of market costs):**

| Year | Revenue (£m) |
|---|---|
| 2022 | 134.9 |
| 2023 | 188.3 |
| 2024 | 107.5 |
| 2025 | 109.4 |

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats as sp_stats

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False,
    'axes.spines.right': False, 'axes.grid': True,
    'grid.linewidth': 0.3, 'grid.alpha': 0.5, 'font.size': 10,
})

IC_DIR     = Path('/Users/aadesh/Documents/IC')
EXCEL_PATH = IC_DIR / 'Cleaned_GBFR.xlsx'
OUTPUT_DIR = IC_DIR / 'M2'
OUTPUT_DIR.mkdir(exist_ok=True)

FID_YEAR        = 2028
REGIME_YEARS    = 25
CAPACITY_MW     = 1_000
AVAILABILITY    = 0.9659
N_PATHS         = 10_000
RANDOM_SEED     = 42
PROJ_TAU_GROWTH = 0.0   # flat nominal; set to non-zero for scenario analysis
ANCHOR_DATE     = pd.Timestamp('2025-01-01')  # tau frozen here (mid 2024-2025 normalised period)

# Ofgem-disclosed IFA2 gross congestion revenues net of market costs (£m)
ACTUALS = {2022: 134.9, 2023: 188.3, 2024: 107.5, 2025: 109.4}

print(f'Config: FID={FID_YEAR}  capacity={CAPACITY_MW} MW  '
      f'availability={AVAILABILITY}  N_paths={N_PATHS:,}')

In [ ]:
SHORT_GAP_H = 4

def _wide_to_hourly(wide_df, value_col):
    df = wide_df.rename(columns={'Unnamed: 0': 'date'}).copy()
    df['date'] = pd.to_datetime(df['date'])
    hour_cols = sorted([c for c in df.columns if c.startswith('H') and c[1:].isdigit()],
                       key=lambda x: int(x[1:]))
    melted = df.melt(id_vars='date', value_vars=hour_cols,
                     var_name='hour_col', value_name=value_col)
    melted['offset']    = melted['hour_col'].str[1:].astype(int) - 1
    melted['timestamp'] = melted['date'] + pd.to_timedelta(melted['offset'], unit='h')
    return melted.set_index('timestamp')[value_col].sort_index().astype(float)

xl       = pd.ExcelFile(EXCEL_PATH)
gb_wide  = xl.parse('GB_FINAL')
fr_wide  = xl.parse('FR_FINAL')
fx_daily = xl.parse('GBP-EUR')

gb_s = _wide_to_hourly(gb_wide, 'gb_price_gbp')
fr_s = _wide_to_hourly(fr_wide, 'fr_price_eur')

fx = fx_daily.copy()
fx.columns = ['date', 'fx_eur_gbp']
fx['date'] = pd.to_datetime(fx['date'])
fx = fx.dropna(subset=['fx_eur_gbp']).set_index('date').sort_index()
fx = fx.reindex(pd.date_range(fx.index.min(), fx.index.max(), freq='D')).ffill()

panel = pd.concat([gb_s, fr_s], axis=1).sort_index()
panel = panel[~panel.index.duplicated(keep='last')]
start = max(panel['gb_price_gbp'].first_valid_index(),
            panel['fr_price_eur'].first_valid_index())
end   = min(panel['gb_price_gbp'].last_valid_index(),
            panel['fr_price_eur'].last_valid_index())
panel = panel.loc[start:end]
panel = panel.reindex(pd.date_range(panel.index.min(), panel.index.max(), freq='h'))
panel = panel.interpolate(method='linear', limit=SHORT_GAP_H).dropna()
panel['fx_eur_gbp']   = pd.Series(panel.index.normalize().map(fx['fx_eur_gbp']), index=panel.index).bfill().ffill()
panel['fr_price_gbp'] = panel['fr_price_eur'] * panel['fx_eur_gbp']
panel['spread_gbp']   = panel['gb_price_gbp'] - panel['fr_price_gbp']
panel = panel.dropna(subset=['gb_price_gbp', 'fr_price_gbp', 'spread_gbp'])

daily      = panel[['gb_price_gbp', 'fr_price_gbp', 'spread_gbp']].resample('D').mean().dropna()
_sample_end = daily.index.max()

print(f'Panel: {panel.index.min().date()} -> {panel.index.max().date()}  ({len(panel):,} hourly)')
print(f'Daily: {daily.index.min().date()} -> {_sample_end.date()}  ({len(daily):,} days)')
print(f'Mean spread: {daily["spread_gbp"].mean():.2f} £/MWh')
print(f'Mean |spread|: {daily["spread_gbp"].abs().mean():.2f} £/MWh')

## Step 3′ — OLS on Spread Levels

Unlike Abadie, we do **not** take logs — the spread can be negative (FR > GB), so log is undefined.
Regress daily spread directly on Fourier + trend + weekend features:

    S_t = b0 + b1·t + b2·sin(2πt) + b3·cos(2πt) + b4·sin(4πt) + b5·cos(4πt) + b6·D_t + ε_t

**Tau fix:** same as Methodology 1 — freeze tau at the sample-end value so the projection
anchors at 2026 spread levels, not at the energy-crisis-distorted trend slope.

In [ ]:
def build_features_spread(idx, t0):
    tau = (idx - t0).total_seconds() / (365.25 * 24 * 3600)
    return pd.DataFrame({
        'const':   1.0,
        'tau':     tau,
        'sin1':    np.sin(2 * np.pi * tau),  # annual — significant (t=+8.5)
        'cos1':    np.cos(2 * np.pi * tau),  # annual — retained jointly
        'weekend': (idx.dayofweek >= 5).astype(float),  # semi-annual dropped (p=0.09, 0.85)
    }, index=idx)

t0_spread    = daily.index[0]
feat_spread  = build_features_spread(daily.index, t0_spread)
res_ols      = sm.OLS(daily['spread_gbp'].values, feat_spread).fit()
f_spread     = pd.Series(res_ols.fittedvalues, index=daily.index)
resid_s      = daily['spread_gbp'] - f_spread
params_s     = pd.Series(res_ols.params, index=feat_spread.columns)
_tau_end_s   = float((_sample_end - t0_spread).total_seconds()) / (365.25 * 24 * 3600)

print('Spread OLS (levels, £/MWh):')
print(f'  R2={res_ols.rsquared:.4f}  N={len(daily):,}  sigma_resid={resid_s.std():.4f} £/MWh')
print(f'  tau_est={params_s["tau"]:.4f} £/MWh/yr  ({params_s["tau"]:.2f} £/MWh drift/yr)')
print(f'  tau_end={_tau_end_s:.3f} yrs  -- anchoring projection here')
print(f'  Deterministic spread at sample end (last 30 days): {f_spread.iloc[-30:].mean():.2f} £/MWh')

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
axes[0].plot(daily.index, daily['spread_gbp'], lw=0.7, alpha=0.7, color='steelblue', label='Actual')
axes[0].plot(daily.index, f_spread, lw=1.5, color='darkorange', label='OLS f_S(t)')
axes[0].axhline(0, color='k', lw=0.8, ls='--', alpha=0.4)
axes[0].set_ylabel('£/MWh'); axes[0].legend()
axes[0].set_title('Fig S1a. Spread OLS fit (levels)', fontweight='bold')
axes[1].plot(daily.index, resid_s, lw=0.7, color='#2ca02c', alpha=0.8)
axes[1].axhline(0, color='k', lw=0.8, ls='--', alpha=0.4)
for sig in [1, 2, 3]:
    axes[1].axhline(+sig * resid_s.std(), color='red', lw=0.5, ls=':', alpha=0.5)
    axes[1].axhline(-sig * resid_s.std(), color='red', lw=0.5, ls=':', alpha=0.5)
axes[1].set_ylabel('Residual (£/MWh)')
axes[1].set_title('Fig S1b. OLS residuals epsilon_t  (dotted = ±1/2/3 sigma)', fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figS1_spread_ols.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def cartea_filter(resid, label, thresh=3.0):
    clean = resid.copy()
    for _ in range(20):
        mu, sig = clean.mean(), clean.std()
        is_jump = (clean - mu).abs() > thresh * sig
        if not is_jump.any():
            break
        clean[is_jump] = np.nan
    jumps = resid[resid.index.isin(clean[clean.isna()].index)]
    clean = clean.dropna()
    print(f'  {label}: {len(jumps)} jumps / {len(resid)} days  sigma_clean={clean.std():.4f} £/MWh')
    return clean, jumps

def build_jump_params(jump_sizes, all_days):
    pos_j = jump_sizes[jump_sizes > 0]
    neg_j = jump_sizes[jump_sizes < 0].abs()
    result = {}
    for sign, jmp in [('pos', pos_j), ('neg', neg_j)]:
        if len(jmp) == 0:
            result[sign] = {'lam_by_q': {q: 0.0 for q in [1,2,3,4]}, 'beta': 0.0, 'n': 0}
            continue
        beta  = float(jmp.mean())
        lam_q = {}
        for q in [1, 2, 3, 4]:
            q_days  = all_days[pd.DatetimeIndex(all_days).quarter == q]
            q_jumps = jmp.index[pd.DatetimeIndex(jmp.index).quarter == q]
            lam_q[q] = len(q_jumps) / max(len(q_days), 1)
        result[sign] = {'lam_by_q': lam_q, 'beta': beta, 'n': len(jmp)}
    return result

print('Step 4: Jump detection on spread residuals:')
resid_s_clean, jumps_s = cartea_filter(resid_s, 'Spread')
jump_params_s = build_jump_params(jumps_s, daily.index)

print('\nJump parameters (in £/MWh, not log-units):')
for sign in ['pos', 'neg']:
    jp = jump_params_s[sign]
    print(f'  {sign}: n={jp["n"]}  mean_lam={sum(jp["lam_by_q"].values())/4:.5f}/day  '
          f'beta={jp["beta"]:.4f} £/MWh  '
          f'Q_lams={[round(jp["lam_by_q"][q], 5) for q in [1,2,3,4]]}')

In [ ]:
def ou_ar1_levels(series, label):
    y   = series.values
    X   = sm.add_constant(y[:-1])
    res = sm.OLS(y[1:], X).fit()
    c, phi  = float(res.params[0]), float(res.params[1])
    eps     = pd.Series(res.resid, index=series.index[1:])
    sigma_d = float(eps.std())
    kappa   = -np.log(max(phi, 1e-9)) * 365.25
    theta   = c / (1 - phi) if abs(1 - phi) > 1e-9 else 0.0
    print(f'  {label}: phi={phi:.5f}  kappa={kappa:.2f}/yr  theta={theta:.4f} £/MWh  sigma_d={sigma_d:.4f} £/MWh')
    return {'phi': phi, 'c': c, 'sigma_d': sigma_d,
            'kappa_yr': kappa, 'theta': theta, 'eps': eps}

print('Step 5: OU on jump-cleaned spread residuals (levels):')
ou_s = ou_ar1_levels(resid_s_clean, 'Spread')

eps = ou_s['eps']
print(f'\nOU residual diagnostics:')
print(f'  mean={eps.mean():.4f}  std={eps.std():.4f}  '
      f'skew={sp_stats.skew(eps):.3f}  excess_kurt={sp_stats.kurtosis(eps):.3f}')

print(f'\nContext vs Methodology 1 (log-price separate OUs):')
print(f'  M1 kappa_GB=97.8/yr, kappa_FR=113.8/yr, sigma_d_GB=0.240, sigma_d_FR=0.421 (log-units)')
print(f'  M2 kappa_spread={ou_s["kappa_yr"]:.1f}/yr, sigma_d_spread={ou_s["sigma_d"]:.4f} £/MWh')

In [ ]:
# ── Projection dates ──────────────────────────────────────────────────────
proj_dates = pd.date_range(
    start=f'{FID_YEAR}-01-01', end=f'{FID_YEAR + REGIME_YEARS - 1}-12-31', freq='D'
)
proj_years = proj_dates.year.values

# ── Project f_S(t) — tau anchored at 2024-2025 annual average ─────────────
_tau_anchor_s = float((ANCHOR_DATE - t0_spread).total_seconds()) / (365.25 * 24 * 3600)

def project_f_spread(params_s, proj_dates, t0, tau_anchor, tau_growth=PROJ_TAU_GROWTH):
    feat = build_features_spread(proj_dates, t0)
    secs_from_anchor = np.array((proj_dates - ANCHOR_DATE).total_seconds(), dtype=float)
    yrs_from_anchor  = secs_from_anchor / (365.25 * 24 * 3600)
    feat = feat.copy()
    feat['tau'] = tau_anchor + tau_growth * np.clip(yrs_from_anchor, 0, None)
    return feat.values @ params_s.values

f_s_proj = project_f_spread(params_s, proj_dates, t0_spread, _tau_anchor_s)
print(f'Projected f_S: yr1 mean={f_s_proj[:365].mean():.2f} £/MWh  '
      f'yr25 mean={f_s_proj[-365:].mean():.2f} £/MWh')

# ── Capture ratio — 2024-2025 normalised period only ─────────────────────
# Exclude 2023 (anomalous — FR nuclear recovery drove above-normal spreads).
# Use 2024-2025 as the representative post-crisis steady state.
# Use 2024-2025 only — 2023 excluded (anomalous FR nuclear recovery year)
actual_norm_mean = np.mean([v for k, v in ACTUALS.items() if k >= 2024])

spread_2024_25  = daily.loc['2024':'2025', 'spread_gbp'].abs().mean()
theoretical_norm = spread_2024_25 * CAPACITY_MW * 24 * 365 * AVAILABILITY / 1e6
capture_ratio   = actual_norm_mean / theoretical_norm

print(f'\nCapture ratio (2024-2025 normalised):')
print(f'  mean(|spread|) 2024-2025: {spread_2024_25:.2f} £/MWh')
print(f'  Theoretical annual rev:   £{theoretical_norm:.1f}m')
print(f'  Actual mean 2024-2025:    £{actual_norm_mean:.1f}m')
print(f'  Capture ratio:            {capture_ratio:.4f}  ({capture_ratio*100:.1f}%)')
print(f'  => ~{(1-capture_ratio)*100:.0f}% deducted as market-related costs')

In [ ]:
# ── Monte Carlo simulation ────────────────────────────────────────────────
rng    = np.random.default_rng(RANDOM_SEED)
BATCH  = 500
n_days = len(proj_dates)
phi     = ou_s['phi'];  c_ou = ou_s['c'];  sigma_d = ou_s['sigma_d']
jp_pos  = jump_params_s['pos'];  jp_neg = jump_params_s['neg']

annual_rev = np.zeros((N_PATHS, REGIME_YEARS))

for b0 in range(0, N_PATHS, BATCH):
    b1 = min(b0 + BATCH, N_PATHS);  n = b1 - b0
    X  = np.zeros(n)
    batch_rev = np.zeros((REGIME_YEARS, n))

    for d in range(n_days):
        eps_d = rng.standard_normal(n)
        X = c_ou + phi * X + sigma_d * eps_d
        q = proj_dates[d].quarter
        for jp, sign in [(jp_pos, +1.0), (jp_neg, -1.0)]:
            lam = jp['lam_by_q'][q]
            if lam > 0 and jp['beta'] > 0:
                X += sign * (rng.random(n) < lam) * rng.exponential(jp['beta'], n)
        S_d   = f_s_proj[d] + X
        rev_d = np.abs(S_d) * CAPACITY_MW * 24.0 * AVAILABILITY / 1e6 * capture_ratio
        yr_idx = proj_years[d] - FID_YEAR
        if 0 <= yr_idx < REGIME_YEARS:
            batch_rev[yr_idx] += rev_d
    annual_rev[b0:b1] = batch_rev.T

    if (b1 % 2500 == 0) or b1 == N_PATHS:
        print(f'  {b1}/{N_PATHS} paths  --  Yr1 P50={np.percentile(annual_rev[:b1,0],50):.0f} £m')

print('\nSimulation complete.')
print(f' {"Yr":>3}  {"Cal":>4}  {"P10":>7}  {"P50":>7}  {"P90":>7}  £m')
for yr in [0, 4, 9, 14, 24]:
    print(f' {yr+1:>3}  {FID_YEAR+yr:>4}  '
          f'{np.percentile(annual_rev[:,yr],10):>7.1f}  '
          f'{np.percentile(annual_rev[:,yr],50):>7.1f}  '
          f'{np.percentile(annual_rev[:,yr],90):>7.1f}')

In [ ]:
p10 = np.percentile(annual_rev, 10, axis=0)
p50 = np.percentile(annual_rev, 50, axis=0)
p90 = np.percentile(annual_rev, 90, axis=0)

rev_df = pd.DataFrame({
    'cal_year':  [FID_YEAR + i for i in range(REGIME_YEARS)],
    'regime_yr': range(1, REGIME_YEARS + 1),
    'p10_gbpm':  p10.round(2), 'p50_gbpm': p50.round(2), 'p90_gbpm': p90.round(2),
})
rev_df.to_csv(OUTPUT_DIR / 'mc_revenue_spread_direct.csv', index=False)
print('Revenue projections saved: mc_revenue_spread_direct.csv')
print(rev_df.to_string(index=False))

# Fan chart + comparison
cal_years = [FID_YEAR + i for i in range(REGIME_YEARS)]
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
ax.fill_between(cal_years, p10, p90, alpha=0.25, color='steelblue', label='P10-P90')
ax.plot(cal_years, p50, lw=2.0, color='steelblue', label='P50')
ax.plot(cal_years, p10, lw=0.9, ls='--', color='steelblue')
ax.plot(cal_years, p90, lw=0.9, ls='--', color='steelblue')
for yr, val in ACTUALS.items():
    ax.scatter([yr], [val], color='red', zorder=5, s=50)
ax.scatter([], [], color='red', label='Ofgem actuals 2022-25')
ax.set_title('M2: Spread-direct OU\n(capture ratio applied)', fontweight='bold')
ax.set_xlabel('Year'); ax.set_ylabel('£m nominal'); ax.legend(fontsize=9)

ax2 = axes[1]
try:
    m1 = pd.read_csv(IC_DIR / 'M1' / 'mc_revenue_projections.csv')
    ax2.fill_between(m1['cal_year'], m1['p10_gbpm'], m1['p90_gbpm'],
                     alpha=0.12, color='darkorange')
    ax2.plot(m1['cal_year'], m1['p50_gbpm'], lw=2, color='darkorange', label='M1 P50 (log-price OU)')
    ax2.plot(m1['cal_year'], m1['p10_gbpm'], lw=0.9, ls='--', color='darkorange')
    ax2.plot(m1['cal_year'], m1['p90_gbpm'], lw=0.9, ls='--', color='darkorange')
except FileNotFoundError:
    pass
ax2.fill_between(cal_years, p10, p90, alpha=0.2, color='steelblue')
ax2.plot(cal_years, p50, lw=2, color='steelblue', label='M2 P50 (spread-direct OU)')
ax2.plot(cal_years, p10, lw=0.9, ls='--', color='steelblue')
ax2.plot(cal_years, p90, lw=0.9, ls='--', color='steelblue')
for yr, val in ACTUALS.items():
    ax2.scatter([yr], [val], color='red', zorder=5, s=50)
ax2.scatter([], [], color='red', label='Ofgem actuals 2022-25')
ax2.set_title('M1 vs M2 comparison\n(dashes = P10/P90)', fontweight='bold')
ax2.set_xlabel('Year'); ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figS2_methodology_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figS2_methodology_comparison.png')

In [ ]:
import shutil, openpyxl

EXCEL_TEMPLATE = IC_DIR / 'copy_cap_and_floor_financial_model_-_ifa2_fpa.xlsm'
INPUT_ROW, COL_YEAR_1 = 25, 22

for pct_label, values in [('p10', p10), ('p50', p50), ('p90', p90)]:
    out_path = OUTPUT_DIR / f'ic_cap_floor_m2_{pct_label}.xlsm'
    shutil.copy2(EXCEL_TEMPLATE, out_path)
    wb = openpyxl.load_workbook(out_path, keep_vba=True)
    ws = wb['Input']
    for i, val in enumerate(values):
        ws.cell(row=INPUT_ROW, column=COL_YEAR_1 + i, value=round(float(val), 2))
    wb.save(out_path); wb.close()
    print(f'Written: {out_path.name}  (yr1={values[0]:.1f}, yr25={values[-1]:.1f} £m)')

print('\nOpen ic_cap_floor_m2_*.xlsm in Excel to evaluate cap/floor breach.')